In [2]:
import spacy
from spacy.matcher import Matcher

In [3]:
nlp = spacy.load("en_core_web_sm") # carregando o modelo básico
with open("historico_wow.txt", "r") as f:
    text = f.read()
doc = nlp(text)

In [6]:
# buscando todos os substantivos isolados
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "PROPN"}]
matcher.add("PROPER_NOUN", [pattern])

matches = matcher(doc)
print(len(matches)) # total de ocorrências

for match in matches[:10]:
    print(doc[match[1]:match[2]])

270
World
First
Kill
Heroic
Lich
King
Paragon
Raid
Knight
2


In [7]:
# capturar sequências de nomes próprios como "Lich King" e "Icecrown Citadel"
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "PROPN", "OP": "+"}]
matcher.add("PROPER_NOUN", [pattern])

matches = matcher(doc)
print(len(matches))

for match in matches[:10]:
    print(doc[match[1]:match[2]])

358
World
World First
First
World First Kill
First Kill
Kill
Heroic
Lich
Lich King
King


In [8]:
# greedy LONGEST garante que "Lich King" seja capturado inteiro e não em partes
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "PROPN", "OP": "+"}]
matcher.add("PROPER_NOUN", [pattern], greedy="LONGEST")

matches = matcher(doc)
matches.sort(key=lambda x: x[1]) # ordenando por posição no texto
print(len(matches))

for match in matches[:10]:
    print(doc[match[1]:match[2]])

192
World First Kill
Heroic
Lich King
Paragon
Raid
Knight
2 Frost
Unholy
Druid
Feral Tank


In [9]:
# buscando padrões do tipo "jogador usou habilidade" PROPN seguido de VERB
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "PROPN", "OP": "+"}, {"POS": "VERB"}]
matcher.add("PLAYER_ACTION", [pattern], greedy="LONGEST")

matches = matcher(doc)
matches.sort(key=lambda x: x[1])
print(len(matches))

for match in matches[:10]:
    print(doc[match[1]:match[2]])

34
Lich King used
Paragon used
Lich King kill
Shambling Horrors appeared
Lazeil took
Protection Paladin works
Lazeil tanked
Spirits spawn
Sejta turned
Hunters focused


In [11]:
action_verbs = ["stun", "tank", "use", "turn", "focus", "move", "kill", "place", "request", "assign"] # lista de verbos de ação para monitorar

pattern1 = [{"POS": "PROPN", "OP": "+"}, {"POS": "VERB", "LEMMA": {"IN": action_verbs}}]
pattern2 = [{"POS": "PROPN", "OP": "+"}, {"POS": "VERB", "LEMMA": {"IN": action_verbs}}, {"POS": "DET", "OP": "?"}, {"POS": "PROPN", "OP": "+"}]
pattern3 = [{"POS": "NUM"}, {"POS": "PROPN", "OP": "+"}, {"POS": "VERB", "LEMMA": {"IN": action_verbs}}]

matcher = Matcher(nlp.vocab) # inicializa o matcher com o vocabulario
matcher.add("RAID_ACTION", [pattern1, pattern2, pattern3], greedy="LONGEST") # adiciona os padrões e define busca

matches = matcher(doc) # executa a busca de padrões no documento
matches.sort(key=lambda x: x[1]) # ordena os resultados pela ordem
print(len(matches))

for match in matches[:15]: # percorre os primeiros 15 resultados encontrados
    print(doc[match[1]:match[2]]) # imprime o trecho do texto

15
Lich King used
Paragon used
Lich King kill
Lazeil tanked
Sejta turned
Two Hunters focused
Hunters used
Val’kyrs using Hammer
Paladins used Holy Wrath
Death Knights used Anti-Magic Shell
Warlocks used Shadow Protection
Lazeil moved the Lich King
Hunters placed
Retribution Paladin used Divine Shield
Spirit moved
